In [1]:
# Import necessary libraries
import os
import glob
import json

import numpy as np
from tqdm import tqdm


In [2]:
def is_wh_equal(boxes):
    concat_boxes = np.concatenate(boxes)
    return np.all(np.isclose(concat_boxes[:, 2], concat_boxes[0, 2])) & \
        np.all(np.isclose(concat_boxes[:, 3], concat_boxes[0, 3]))


def get_boxes(label_paths, video_names, name): 
    tmp_label_pths = label_paths[np.in1d(video_names, name)]
    boxes = []
    for pth in tmp_label_pths:
        tmp_boxes= []
        with open(pth, "rt") as f:
            for line in f.readlines():
                _, x_c, y_c, w, h, _, _ = [float(i) for i in line.split()]
                tmp_boxes.append([x_c, y_c, w, h])
        boxes.append(tmp_boxes)
    
    return boxes

def is_equal_boxes(label_paths, video_names, name):
    boxes = get_boxes(label_paths, video_names, name)
    return is_wh_equal(boxes)

In [22]:
BASE_PTH = "/home/klymy/data/people_detection/baseline_10sec"
SUBSET = "algotest"
IMGS_PTH = os.path.join(BASE_PTH, "images", SUBSET)
LABELS_PTH = os.path.join(BASE_PTH, "labels", SUBSET)

img_paths = np.array(sorted(glob.glob(os.path.join(IMGS_PTH, "*"))))
label_paths = np.array(sorted(glob.glob(os.path.join(LABELS_PTH, "*"))))

log_pth = f"log_{SUBSET}.json"

video_names = [os.path.basename(i)[:-10] for i in img_paths]
uvideo_names, inverse_mapping = np.unique(video_names, return_inverse=True)

len(uvideo_names), len(video_names)

(477, 9572)

In [23]:
incorrect_boxes = np.array([is_equal_boxes(label_paths, video_names, i) for i in tqdm(uvideo_names)])

sum(incorrect_boxes), sum(~incorrect_boxes)

100%|██████████| 477/477 [00:01<00:00, 344.13it/s]


(0, 477)

In [24]:
filtered_frames_idxs = np.in1d(video_names, uvideo_names[incorrect_boxes])
filtered_frames_idxs.sum(), len(video_names)

(0, 9572)

In [25]:
np.array(label_paths)[~filtered_frames_idxs].shape

(9572,)

In [26]:
with open(f"normal_frames_{SUBSET}.json", "wt") as f:
    json.dump({
        "incorrect_count": int(filtered_frames_idxs.sum()),
        "correct_count": len(img_paths) - int(filtered_frames_idxs.sum()),
        "imgs": np.array(img_paths)[~filtered_frames_idxs].tolist(),
        "labels": np.array(label_paths)[~filtered_frames_idxs].tolist(),
    }, f, indent=4)